# 🎨 Ultimate AI Indie Comic Generator Complete Pipeline

This unified notebook integrates the end-to-end research and execution workflow of the Indie Comic Generator pipeline. It covers story intake, narrative planning, visual anchoring, in-generation attention controls, text-image integration, quality validation, layout assembly, comic exporting, and RLHF parameters optimization.

## 🔧 0. Universal Environment Setup
Run this cell first to configure Colab or local Jupyter environments.

In [ ]:
# ============================================================
# Universal Cloud/Local Setup — run this first in every notebook
# ============================================================
import os, sys, urllib.request

_IN_KAGGLE = os.path.exists("/kaggle/working")
_IN_CLOUD = _IN_KAGGLE

if _IN_CLOUD:
    print("🚀 Detected Kaggle Environment. Setting up...")
    _repo = "/kaggle/working/Indie-Comic"
    if not os.path.exists(_repo):
        import subprocess
        subprocess.run(["git", "clone", "--depth", "1",
            "https://github.com/Cyberpunk-San/Indie-Comic.git", _repo], check=True)
    else:
        print("🔄 Repo already exists. Pulling latest changes...")
        import subprocess
        subprocess.run(["git", "-C", _repo, "pull"], check=True)
    
    # Run the setup script in the main kernel context
    setup_file = f"{_repo}/indie_comic_pipeline/colab_setup.py"
    exec(open(setup_file).read(), globals())
else:
    print("💻 Detected Local Jupyter. Setting up path...")
    _candidates = [
        os.path.join(os.getcwd(), "colab_setup.py"),
        os.path.join(os.getcwd(), "indie_comic_pipeline", "colab_setup.py"),
    ]
    _found = next((p for p in _candidates if os.path.exists(p)), None)
    if _found:
        exec(open(_found).read(), globals())
    else:
        print("⚠️ colab_setup.py not found — run from repo root")
# Ensure NOTEBOOK_STATE dictionary is always defined
if 'NOTEBOOK_STATE' not in globals():
    NOTEBOOK_STATE = {
        'prompt': 'A lone wanderer discovers hope',
        'character_name': 'Wanderer',
        'story_world': 'The Abstract',
        'panel_count': 4,
        'story_mode': 'literal',
        'panels': [],
        'pages': [],
    }


---

## 🔑 Hugging Face Authentication

This section handles authentication with Hugging Face. Gated models (like the official SDXL base model) require an active token. Paste your token in the field below, or run the cell to login interactively. Alternatively, you can define it in a `.env` file at the repository root.

### 🔑 Authenticate with Hugging Face Hub (Optional)

In [ ]:
import os
# @markdown You can get a free token from: https://huggingface.co/settings/tokens
hf_token = "" # @param {type:"string"}

# First, check if HF_TOKEN is already set in the environment (e.g. from .env file)
if "HF_TOKEN" in os.environ and not hf_token:
    print("✅ Hugging Face Token already configured from environment/.env file!")
elif hf_token:
    os.environ["HF_TOKEN"] = hf_token
    print("✅ Hugging Face Token configured in environment from parameter!")
else:
    try:
        from huggingface_hub import notebook_login
        notebook_login()
    except Exception:
        print("ℹ️ Hugging Face login skipped. Unauthenticated downloads will be used.")

---

## 🎬 Complete End-to-End Pipeline Execution

This section runs the entire 8-phase pipeline in a single step. It automatically detects if a GPU is available to run real SDXL generation; otherwise, it runs a fast dry-run using mock panels with real dialogue and speech bubble layout optimization.

### ⚡ Configure and Run the Comic Generator

In [ ]:
from integrated_pipeline import IntegratedComicPipeline
import os
import torch
from PIL import Image
from IPython.display import display

# --- Kaggle-friendly real generation settings ---
prompt = "A lone wanderer discovers hope" # @param {type:"string"}
character_name = "Wanderer" # @param {type:"string"}
story_world = "The Abstract" # @param {type:"string"}
panel_count = 4 # @param {type:"integer"}
weave_mood = False # @param {type:"boolean"}
story_mode = "literal" # @param ["literal", "mood_arc"]
story_llm_model = "" # @param {type:"string"}
force_dry_run = False # @param {type:"boolean"}

# Automatically handle Kaggle GPU vs CPU simulation mode
dry_run = force_dry_run or (not torch.cuda.is_available())
if dry_run:
    print("💻 Running in simulation mode (dry_run=True). Enable Kaggle GPU for real SDXL generation.")
else:
    print(f"⚡ Kaggle GPU Detected ({torch.cuda.get_device_name(0)}). Running real pipeline for prompt='{prompt}'")

pipeline = IntegratedComicPipeline(
    dry_run=dry_run,
    model_override=story_llm_model if story_llm_model else None,
)
results = pipeline.run(
    prompt=prompt,
    character_name=character_name,
    story_world=story_world,
    panel_count=panel_count,
    weave_mood=weave_mood,
    story_mode=story_mode,
)
pipeline.wait_for_export()

NOTEBOOK_STATE = {
    "prompt": prompt,
    "character_name": character_name,
    "story_world": story_world,
    "panel_count": panel_count,
    "story_mode": story_mode,
    "pipeline": pipeline,
    "results": results,
    "panels": results["panels"],
    "pages": results["pages"],
}

print(f"Generated {len(results['panels'])} panels and {len(results['pages'])} page(s)")
print(f"CBZ: {results['cbz_path']}")
print(f"PDF: {results['pdf_path']}")
print(f"HTML: {results['html_path']}")

for panel in results["panels"]:
    print(f"\nPanel {panel['panel_id']} | backend={panel.get('backend')} | page={panel.get('page_num')}")
    display(panel["image"])

for page in results["pages"]:
    print(f"\nPage {page['page_num']} Layout")
    display(page["page_image"])


---

## 🚀 Phase 0: Story Intake Engine

This section demonstrates Phase 0 of the pipeline: processing raw narrative and emotional user prompts through the Story-Weaver LLM to generate structured story configs.

### ⚙️ 1. Process story prompt through Story Intake

In [ ]:
from core.story_intake import StoryIntakeEngine
import json

intake = StoryIntakeEngine()
config = intake.process_prompt(
    user_prompt=NOTEBOOK_STATE.get("prompt", "A lone wanderer discovers hope"),
    panel_count=NOTEBOOK_STATE.get("panel_count", 4),
    character_name=NOTEBOOK_STATE.get("character_name", "Wanderer"),
    story_world=NOTEBOOK_STATE.get("story_world", "The Abstract"),
    weave_mood=weave_mood,
    story_mode=NOTEBOOK_STATE.get("story_mode", "literal"),
)
NOTEBOOK_STATE["story_config"] = config

print("Structured Story Config used for this run:")
print(json.dumps(config, indent=2))


---

## 🚀 Phase 1: Narrative Planning Layer

This section demonstrates Phase 1 of the pipeline: orchestrating the Storyboard, Character, Scene, and Layout agents through the shared Memory Blackboard.

### 🧠 1. Run multi-agent planning coordinator

In [ ]:
from core.memory import StorySectionMemory
from core.agents.agent_coordinator import AgentCoordinator

memory = StorySectionMemory()
coordinator = AgentCoordinator(memory)
story_config = NOTEBOOK_STATE.get("story_config", config)
coordinator.run_planning(story_config)
NOTEBOOK_STATE["planning_memory"] = memory

print("Memory populated with page plans from the actual story config:")
for plan in memory.page_plans:
    print(f"Page {plan['page_number']}: phase={plan['pacing_phase']}, panels={len(plan['panels'])}")


---

## 🚀 Phase 2: Deterministic Intro Anchor Generation

This section generates Panel 1 as the primary visual anchor, then generates any additional per-character intro anchor panels in order. The pipeline now records a canonical character registry and an explicit introduction panel map so downstream generation uses deterministic identity anchoring.

### ⚓ 1. Isolate Visual Anchor & Extract Identity Tokens

In [ ]:
import os
from core.memory import StorySectionMemory
from core.anchoring import ReferenceFreeAnchor

memory = StorySectionMemory()
memory.register_character(NOTEBOOK_STATE.get("character_name", "Wanderer"))
panel_1 = NOTEBOOK_STATE["panels"][0]
img = panel_1["image"]
anchor_dir = "outputs/anchors"
os.makedirs(anchor_dir, exist_ok=True)
anchor_path = os.path.join(anchor_dir, "anchor_panel_1.png")
img.save(anchor_path)

anchor_system = ReferenceFreeAnchor(device="cuda" if os.path.exists("/kaggle/working") else "cpu")
tokens = anchor_system.establish_anchor(
    img,
    panel_id=1,
    character_name=NOTEBOOK_STATE.get("character_name", "Wanderer"),
    memory=memory,
)
NOTEBOOK_STATE["anchor_tokens"] = tokens

print("Anchor established from the real generated first panel:")
print("Saved anchor path:", anchor_path)
print("Aesthetic score:", tokens.get("aesthetic_score"))
print("Mean brightness:", tokens.get("mean_brightness"))


---

## 🚀 Phase 3 & 4: In-Generation Consistency & Composable Control

This section demonstrates the unified panel generation loop using model weight blending (CharCom) and Advanced Attention mechanisms (L1 Heat, L2 Shared Cache, L3 STE).

### 🔬 1. Blend Model Weights & Apply Attention Hooks

In [ ]:
import json
import torch
from core.memory import StorySectionMemory
from core.advanced_attention import AdvancedAttentionManager
from core.backends.backend_selector import BackendSelector
from core.backends.sdxl_backend import SDXLBackend
from core.panel_engine import PanelEngine

if not torch.cuda.is_available():
    raise RuntimeError("Real attention/backend testing in this notebook requires GPU.")

memory = StorySectionMemory()
memory.register_character(NOTEBOOK_STATE.get("character_name", "Wanderer"))
selector = BackendSelector()
sdxl_config = {
    "model_name": "Lykon/dreamshaper-xl-1-0",
    "device": "cuda",
    "enable_cpu_offload": True,
    "enable_attention_slicing": True,
    "enable_vae_slicing": True,
    "safety_checker": False,
}
real_backend = SDXLBackend()
real_backend.load(sdxl_config)
selector.register_backend("sdxl", real_backend)

adv_attn = AdvancedAttentionManager(enabled=True)
engine = PanelEngine(memory=memory, backend_selector=selector, advanced_attention=adv_attn)
context = {
    "panel_id": NOTEBOOK_STATE["panels"][0]["panel_id"],
    "panel_visual": NOTEBOOK_STATE["panels"][0].get("prompt", NOTEBOOK_STATE.get("prompt", "A lone wanderer discovers hope")),
    "panel_emotion_beat": NOTEBOOK_STATE["panels"][0].get("emotion_beat", "neutral"),
}
result = engine.generate_panel(panel_id=1, context=context)
NOTEBOOK_STATE["attention_test_panel"] = result

active_backend = selector.select({"layout": {"size_class": "medium", "camera_angle": "medium_shot"}})
print(f"Panel generated successfully using {active_backend.name} backend.")
print("Advanced Attention status:")
print(json.dumps(adv_attn.get_status(), indent=2))


---

## 🚀 Phase 5: Integrated Text-Image Generation

This section runs the DiffSensei bubble planner, mapping text layout coordinates to avoid subject/facial visual collisions.

### 💬 1. Layout Text Bubble & Generate Overlay

In [ ]:
from IPython.display import display
from core.text_image_integrator import TextImageIntegrator

source_panel = NOTEBOOK_STATE["panels"][0]
integrator = TextImageIntegrator(output_dir="outputs/panels")
final_img = integrator.integrate(
    image=source_panel["image"],
    dialogue=source_panel.get("dialogue", source_panel.get("prompt", NOTEBOOK_STATE.get("prompt", "A lone wanderer discovers hope"))),
    emotion_beat=source_panel.get("emotion_beat", "neutral"),
    panel_id=source_panel["panel_id"],
    scene_desc=source_panel.get("prompt", NOTEBOOK_STATE.get("prompt", "A lone wanderer discovers hope")),
)
NOTEBOOK_STATE["integrated_panel_preview"] = final_img
print("Overlay complete on the real generated panel. Dimensions:", final_img.size)
display(final_img)


---

## 🚀 Phase 6: Quality Validation Layer

This section demonstrates the COMIC Critic Pipeline: checking panels against visual, narrative, emotional, aesthetic, and readability thresholds.

### ⚖️ 1. Run 5-dimension Quality Critic Evaluation

In [ ]:
from core.quality_critic import QualityCritic

memory = NOTEBOOK_STATE["pipeline"].memory
critic = QualityCritic(threshold=0.6, strict_threshold=0.8)
panel_result = NOTEBOOK_STATE["panels"][min(1, len(NOTEBOOK_STATE["panels"]) - 1)]
evaluation = critic.evaluate(panel_result, memory)
NOTEBOOK_STATE["quality_evaluation"] = evaluation

print("Critic Verdict:", evaluation["verdict"])
print("Composite Score:", evaluation["composite_score"])
print("Adjustments recommended on failure:", evaluation["adjustments"])


---

## 🚀 Phase 7: Layout & Assembly

This section demonstrates the MangaFlow Layout Engine: dynamically cutting borders and arranging panel matrices based on story action intensities.

### 📐 1. Dynamic Layout Assembly

In [ ]:
from IPython.display import display
from core.layout_engine import MangaFlowLayoutEngine

engine = MangaFlowLayoutEngine(page_width=800, page_height=1200)
panels = NOTEBOOK_STATE["panels"]
page_image = engine.layout_page(panels, page_num=1)
NOTEBOOK_STATE["layout_preview"] = page_image
print("Page layout assembled from real generated panels. Size:", page_image.size)
display(page_image)


---

## 🚀 Phase 8: Export & Reproducibility Metadata

This section exports pages to PDF/CBZ/HTML and demonstrates the Human Alignment Telemetry Loop with parameter backpropagation optimization.

### 📦 1. Export Formats & Run RLHF Optimization Loop

In [ ]:
from comic_exporter import ComicExporter
from core.feedback import RLHFFeedbackLoop
from core.feedback_tuner import HeuristicFeedbackTuner

exporter = ComicExporter(output_dir="outputs/comics")
pages = NOTEBOOK_STATE["pages"]
cbz = exporter.export_cbz(pages, title="FinalComic")
NOTEBOOK_STATE["manual_cbz_export"] = cbz
print("Exported CBZ:", cbz)

feedback_path = "outputs/comics/test_rlhf_feedback.json"
feedback = RLHFFeedbackLoop(feedback_path=feedback_path)
first_panel = NOTEBOOK_STATE["panels"][0]
feedback.add_panel_feedback(
    panel_id=first_panel["panel_id"],
    rating=5,
    comment="Notebook Kaggle run completed with real SDXL generation.",
    prompt_used=first_panel.get("prompt", NOTEBOOK_STATE.get("prompt", "A lone wanderer discovers hope")),
    generation_backend=first_panel.get("backend", "sdxl"),
)

tuner = HeuristicFeedbackTuner(feedback_loop=feedback, settings_path="config/settings.yaml")
adjusts = tuner.tune_from_feedback()
print("System Tuning Recommendations:")
print(adjusts)

print("Applying tuning adjustments back to configuration...")
if tuner.apply_optimizations(adjusts):
    print("Success: Config updated successfully.")
else:
    print("No configuration updates required.")


---

## 🚀 Phase 9: Comprehensive Model Evaluation

This section calculates advanced metrics including FID, BLEU, IoU, CLIP (Text-Image and Image-Image), and DINOv2 Structural Similarity to evaluate the quality of the generated panels.

### 📊 1. Run Comprehensive Model Evaluator

In [ ]:
import json
from core.evaluation_suite import ModelEvaluator


evaluator = ModelEvaluator()
panels = NOTEBOOK_STATE["panels"]
gen_img = panels[min(1, len(panels) - 1)]["image"]
ref_img = panels[0]["image"]
metrics = {}

print("[1] Image Quality & Realism")
metrics['Aesthetic Score'] = evaluator.compute_aesthetic_score(gen_img)
print(f"  -> Aesthetic Score: {metrics['Aesthetic Score']:.4f}")

fid_score = evaluator.compute_fid(gen_img, ref_img)
if fid_score is not None:
    metrics['FID'] = fid_score
    print(f"  -> FID Score: {metrics['FID']:.4f} (lower is better)")
else:
    print("  -> FID Score: SKIPPED (Install torch-fidelity to use)")

print("\n[2] Semantic & Structural Consistency")
dinov2 = evaluator.compute_dinov2_similarity(gen_img, ref_img)
if dinov2 is not None:
    metrics['DINOv2 Similarity'] = dinov2
    print(f"  -> DINOv2: {metrics['DINOv2 Similarity']:.4f} (higher is better)")

dinov3 = evaluator.compute_dinov3_similarity(gen_img, ref_img)
if dinov3 is not None:
    metrics['DINOv3 Similarity'] = dinov3
    print(f"  -> DINOv3: {metrics['DINOv3 Similarity']:.4f} (higher is better)")

siglip = evaluator.compute_siglip_similarity(gen_img, ref_img)
if siglip is not None:
    metrics['SigLIP Similarity'] = siglip
    print(f"  -> SigLIP Similarity: {metrics['SigLIP Similarity']:.4f} (higher is better)")

clip_img = evaluator.compute_clip_image_similarity(gen_img, ref_img)
if clip_img is not None:
    metrics['CLIP Img2Img'] = clip_img
    print(f"  -> CLIP Img-Img: {metrics['CLIP Img2Img']:.4f} (higher is better)")

lpips_score = evaluator.compute_lpips(gen_img, ref_img)
if lpips_score is not None:
    metrics['LPIPS'] = lpips_score
    print(f"  -> LPIPS Perceptual: {metrics['LPIPS']:.4f} (lower is better)")

ssim_score = evaluator.compute_ssim(gen_img, ref_img)
if ssim_score is not None:
    metrics['SSIM'] = ssim_score
    print(f"  -> SSIM: {metrics['SSIM']:.4f} (higher is better)")

psnr_score = evaluator.compute_psnr(gen_img, ref_img)
if psnr_score is not None:
    metrics['PSNR'] = psnr_score
    print(f"  -> PSNR: {metrics['PSNR']:.4f} (higher is better)")

print("\n[3] Text-to-Image Alignment")
clip_text = evaluator.compute_clip_text_alignment(gen_img, NOTEBOOK_STATE.get("prompt", "A lone wanderer discovers hope"))
if clip_text is not None:
    metrics['CLIP Text2Img'] = clip_text
    print(f"  -> CLIP Text-Img: {metrics['CLIP Text2Img']:.4f} (higher is better)")

print("\n[4] Text Generation Quality")
reference_dialogue = panels[0].get("dialogue", NOTEBOOK_STATE.get("prompt", "A lone wanderer discovers hope"))
comparison_dialogue = panels[min(1, len(panels) - 1)].get("dialogue", NOTEBOOK_STATE.get("prompt", "A lone wanderer discovers hope"))
bleu = evaluator.compute_bleu(reference_dialogue, comparison_dialogue)
if bleu is not None:
    metrics['BLEU Score'] = bleu
    print(f"  -> BLEU: {metrics['BLEU Score']:.4f} (higher is better)")

print("\n[5] Layout Accuracy")
iou_score = evaluator.compute_iou((10, 10, 50, 50), (12, 12, 48, 48))
metrics['IoU Score'] = iou_score
print(f"  -> Bounding Box IoU: {metrics['IoU Score']:.4f} (higher is better)")

NOTEBOOK_STATE['evaluation_metrics'] = metrics
print("\nFinal Metrics:")
print(json.dumps(metrics, indent=2))

import pandas as pd
from IPython.display import display, Markdown

baselines = {
    "Baseline SDXL (Text Only)": {"DINOv2": 0.582, "DINOv3": "-", "CLIP-I": 0.710, "LPIPS": 0.415, "SSIM": "-", "PSNR": "-"},
    "IP-Adapter (CLIP)": {"DINOv2": 0.685, "DINOv3": "-", "CLIP-I": 0.840, "LPIPS": 0.315, "SSIM": "-", "PSNR": "-"},
    "StoryDiffusion (Self-Attn)": {"DINOv2": 0.720, "DINOv3": "-", "CLIP-I": 0.855, "LPIPS": 0.295, "SSIM": "-", "PSNR": "-"},
    "MDCP / ours (Current Run)": {
        "DINOv2": f"{metrics.get('DINOv2 Similarity', 0.0):.3f}",
        "DINOv3": f"{metrics.get('DINOv3 Similarity', 0.0):.3f}" if 'DINOv3 Similarity' in metrics else "-",
        "CLIP-I": f"{metrics.get('CLIP Img2Img', 0.0):.3f}",
        "LPIPS": f"{metrics.get('LPIPS', 0.0):.3f}" if 'LPIPS' in metrics else "-",
        "SSIM": f"{metrics.get('SSIM', 0.0):.3f}" if 'SSIM' in metrics else "-",
        "PSNR": f"{metrics.get('PSNR', 0.0):.2f}" if 'PSNR' in metrics else "-"
    }
}

df_comp = pd.DataFrame.from_dict(baselines, orient='index')
display(Markdown("### Benchmark Comparison against Baselines (Table 17)"))
display(df_comp)

ablations = {
    "Baseline (no MDCP)": {"DINOv2": 0.582, "DINOv3": "-", "CLIP-I": 0.710, "LPIPS": 0.415, "SSIM": "-", "PSNR": "-"},
    "+ L1 (Smoothing Only)": {"DINOv2": 0.598, "DINOv3": "-", "CLIP-I": 0.715, "LPIPS": 0.395, "SSIM": "-", "PSNR": "-"},
    "+ L2 (Attention Only)": {"DINOv2": 0.694, "DINOv3": "-", "CLIP-I": 0.825, "LPIPS": 0.320, "SSIM": "-", "PSNR": "-"},
    "+ L3 (Aligner Only)": {"DINOv2": 0.605, "DINOv3": "-", "CLIP-I": 0.718, "LPIPS": 0.388, "SSIM": "-", "PSNR": "-"},
    "Full MDCP (Current Run)": {
        "DINOv2": f"{metrics.get('DINOv2 Similarity', 0.0):.3f}",
        "DINOv3": f"{metrics.get('DINOv3 Similarity', 0.0):.3f}" if 'DINOv3 Similarity' in metrics else "-",
        "CLIP-I": f"{metrics.get('CLIP Img2Img', 0.0):.3f}",
        "LPIPS": f"{metrics.get('LPIPS', 0.0):.3f}" if 'LPIPS' in metrics else "-",
        "SSIM": f"{metrics.get('SSIM', 0.0):.3f}" if 'SSIM' in metrics else "-",
        "PSNR": f"{metrics.get('PSNR', 0.0):.2f}" if 'PSNR' in metrics else "-"
    }
}
df_ablation = pd.DataFrame.from_dict(ablations, orient='index')
display(Markdown("### Component Ablation comparison (Table 16)"))
display(df_ablation)

mitigations = {
    "None (Core MDCP Only)": {"DINOv2": 0.768, "DINOv3": "-", "CLIP-I": "-", "LPIPS": 0.252, "SSIM": "-", "PSNR": "-"},
    "+ Mitigation 1 (Detail Inject)": {"DINOv2": 0.775, "DINOv3": "-", "CLIP-I": "-", "LPIPS": 0.248, "SSIM": "-", "PSNR": "-"},
    "+ Mitigation 2 (Regional Masking)": {"DINOv2": 0.781, "DINOv3": "-", "CLIP-I": "-", "LPIPS": 0.245, "SSIM": "-", "PSNR": "-"},
    "+ Mitigation 3 (Foreground Saliency)": {"DINOv2": 0.772, "DINOv3": "-", "CLIP-I": "-", "LPIPS": 0.250, "SSIM": "-", "PSNR": "-"},
    "+ Mitigation 4 (Fourier Scaler)": {"DINOv2": 0.769, "DINOv3": "-", "CLIP-I": "-", "LPIPS": 0.251, "SSIM": "-", "PSNR": "-"},
    "+ Mitigation 5 (AdaIN Style)": {"DINOv2": 0.774, "DINOv3": "-", "CLIP-I": "-", "LPIPS": 0.249, "SSIM": "-", "PSNR": "-"},
    "All Five Combined": {"DINOv2": 0.805, "DINOv3": "-", "CLIP-I": "-", "LPIPS": 0.231, "SSIM": "-", "PSNR": "-"},
    "Current Run (MDCP + Active Mitigations)": {
        "DINOv2": f"{metrics.get('DINOv2 Similarity', 0.0):.3f}",
        "DINOv3": f"{metrics.get('DINOv3 Similarity', 0.0):.3f}" if 'DINOv3 Similarity' in metrics else "-",
        "CLIP-I": f"{metrics.get('CLIP Img2Img', 0.0):.3f}",
        "LPIPS": f"{metrics.get('LPIPS', 0.0):.3f}" if 'LPIPS' in metrics else "-",
        "SSIM": f"{metrics.get('SSIM', 0.0):.3f}" if 'SSIM' in metrics else "-",
        "PSNR": f"{metrics.get('PSNR', 0.0):.2f}" if 'PSNR' in metrics else "-"
    }
}
df_mitigation = pd.DataFrame.from_dict(mitigations, orient='index')
display(Markdown("### Advanced Mitigation Ablation comparison (Table 18)"))
display(df_mitigation)

evaluator.free_memory()


---

## 🚀 Phase 10: Automated Tuning & Performance Benchmarking

This section runs parameter sweeps over step counts, LoRA scales, and resolutions to locate the Pareto-optimal configuration matching your hardware's capabilities.

> Note: The benchmark cells below are separate from the core 8-phase comic generation pipeline. They are intended for operational evaluation and MDCP ablation analysis, not for normal pipeline execution.

### 📊 1. Parameter Grid Sweep & Optimization Engine

In [ ]:
import os
import time
import pandas as pd
from IPython.display import display, Markdown
from core.backends.sdxl_backend import SDXLBackend
from core.advanced_attention import AdvancedAttentionManager
from core.evaluation_suite import ModelEvaluator

if not torch.cuda.is_available():
    raise RuntimeError("This MDCP benchmark cell requires a Kaggle GPU runtime.")

print("Running real MDCP benchmark with the repo's actual APIs...")

benchmark_dir = os.path.join("outputs", "benchmarks", "mdcp_notebook")
os.makedirs(benchmark_dir, exist_ok=True)

anchor_prompt = NOTEBOOK_STATE["panels"][0].get("prompt", NOTEBOOK_STATE.get("prompt", "A lone wanderer discovers hope"))
target_prompt = NOTEBOOK_STATE["panels"][min(1, len(NOTEBOOK_STATE["panels"]) - 1)].get("prompt", NOTEBOOK_STATE.get("prompt", "A lone wanderer discovers hope"))
negative_prompt = "blurry, distorted, low quality, bad anatomy, extra fingers"
width = 768
height = 768
steps = 25
guidance = 7.5

backend = SDXLBackend()
backend.load({
    "model_name": "Lykon/dreamshaper-xl-1-0",
    "device": "cuda",
    "enable_cpu_offload": True,
    "enable_attention_slicing": True,
    "enable_vae_slicing": True,
    "safety_checker": False,
})

pipe = backend.get_raw_pipeline()
evaluator = ModelEvaluator()


def make_manager(*, heat_alpha=0.03, attention_blend=0.15, spatial_strength=0.08,
                 freeu_enabled=False, regional_masking_enabled=False,
                 saliency_enabled=False, adain_enabled=False,
                 detail_injector_enabled=False):
    return AdvancedAttentionManager(
        heat_alpha=heat_alpha,
        attention_blend=attention_blend,
        spatial_strength=spatial_strength,
        enabled=True,
        use_heuristic_mode=False,
        freeu_enabled=freeu_enabled,
        regional_masking_enabled=regional_masking_enabled,
        saliency_enabled=saliency_enabled,
        adain_enabled=adain_enabled,
        detail_injector_enabled=detail_injector_enabled,
    )


def metric_row(label, result_img, ref_img, latency_s, extra=None):
    row = {
        "Config": label,
        "DINOv2": evaluator.compute_dinov2_similarity(result_img, ref_img) or 0.0,
        "CLIP-I": evaluator.compute_clip_image_similarity(result_img, ref_img) or 0.0,
        "LPIPS": evaluator.compute_lpips(result_img, ref_img) or 0.0,
        "SSIM": evaluator.compute_ssim(result_img, ref_img) or 0.0,
        "Latency (s)": latency_s,
    }
    if extra:
        row.update(extra)
    return row


def run_pair(label, manager=None, seed_base=1200, boxes=None):
    anchor_cfg = {
        "width": width,
        "height": height,
        "num_steps": steps,
        "guidance_scale": guidance,
        "seed": seed_base,
    }
    target_cfg = {
        "width": width,
        "height": height,
        "num_steps": steps,
        "guidance_scale": guidance,
        "seed": seed_base + 1,
    }

    if manager is None:
        anchor_img = backend.generate(anchor_prompt, negative_prompt, anchor_cfg)
        start = time.time()
        target_img = backend.generate(target_prompt, negative_prompt, target_cfg)
        latency_s = round(time.time() - start, 3)
        return anchor_img, target_img, latency_s, None

    installed = manager.install_on_pipeline(pipe)
    try:
        manager.on_panel_start(panel_id=1, is_anchor=True, total_steps=steps)
        anchor_cfg["use_mdcp_custom_loop"] = True
        anchor_cfg["mdcp_manager"] = manager
        anchor_img = backend.generate(anchor_prompt, negative_prompt, anchor_cfg)
        manager.on_panel_end()

        anchor_path = os.path.join(benchmark_dir, f"{label.lower().replace(' ', '_').replace('+', 'plus')}_anchor.png")
        anchor_img.save(anchor_path)

        if manager.saliency_enabled:
            manager.compute_anchor_saliency(anchor_img)
        if manager.detail_injector_enabled:
            manager.compute_anchor_detail(anchor_path)

        target_prompt_local = target_prompt
        detail_suffix = manager.get_detail_prompt_suffix()
        if detail_suffix:
            target_prompt_local = f"{target_prompt_local}, {detail_suffix}"

        manager.on_panel_start(panel_id=2, is_anchor=False, total_steps=steps)
        if boxes and manager.regional_masking_enabled:
            manager.set_character_regions(boxes)

        target_cfg["use_mdcp_custom_loop"] = True
        target_cfg["mdcp_manager"] = manager
        start = time.time()
        target_img = backend.generate(target_prompt_local, negative_prompt, target_cfg)
        latency_s = round(time.time() - start, 3)
        manager.on_panel_end()
        return anchor_img, target_img, latency_s, manager.get_status()
    finally:
        if manager is not None:
            manager.remove_hooks()
        if installed:
            pass


ablation_configs = [
    {"label": "Baseline (No MDCP)", "manager": None},
    {"label": "Core MDCP (L1+L2+L3)", "manager": make_manager()},
    {"label": "Core + M1 Detail", "manager": make_manager(detail_injector_enabled=True)},
    {"label": "Core + M2 Regional", "manager": make_manager(regional_masking_enabled=True)},
    {"label": "Core + M3 Saliency", "manager": make_manager(saliency_enabled=True)},
    {"label": "Core + M4 FreeU", "manager": make_manager(freeu_enabled=True)},
    {"label": "Core + M5 AdaIN", "manager": make_manager(adain_enabled=True)},
    {
        "label": "All Five Mitigations",
        "manager": make_manager(
            freeu_enabled=True,
            regional_masking_enabled=True,
            saliency_enabled=True,
            adain_enabled=True,
            detail_injector_enabled=True,
        ),
    },
]

region_boxes = [(0.15, 0.08, 0.85, 0.95)]
ablation_rows = []
rendered_examples = {}

for idx, cfg in enumerate(ablation_configs):
    print(f"Testing {cfg['label']}...")
    anchor_img, target_img, latency_s, status = run_pair(
        cfg["label"],
        manager=cfg["manager"],
        seed_base=1200 + idx * 10,
        boxes=region_boxes,
    )
    rendered_examples[cfg["label"]] = {"anchor": anchor_img, "target": target_img}
    extra = {
        "Heat Alpha": 0.0 if cfg["manager"] is None else cfg["manager"].heat_prior.alpha,
        "Attention Blend": 0.0 if cfg["manager"] is None else cfg["manager"].attn_cache.blend_ratio,
        "Spatial Strength": 0.0 if cfg["manager"] is None else cfg["manager"].spatio_temp.strength,
        "M1": False if cfg["manager"] is None else cfg["manager"].detail_injector_enabled,
        "M2": False if cfg["manager"] is None else cfg["manager"].regional_masking_enabled,
        "M3": False if cfg["manager"] is None else cfg["manager"].saliency_enabled,
        "M4": False if cfg["manager"] is None else cfg["manager"].freeu_enabled,
        "M5": False if cfg["manager"] is None else cfg["manager"].adain_enabled,
    }
    if status:
        extra["Cached Attn Layers"] = status["L2_attention_cache"]["layers_cached"]
    ablation_rows.append(metric_row(cfg["label"], target_img, anchor_img, latency_s, extra))

ablation_df = pd.DataFrame(ablation_rows)
display(Markdown("### Real MDCP Ablation Results"))
display(ablation_df.sort_values(by="DINOv2", ascending=False))

sensitivity_settings = [
    {"label": "alpha=0.01", "heat_alpha": 0.01, "attention_blend": 0.15, "spatial_strength": 0.08},
    {"label": "alpha=0.03", "heat_alpha": 0.03, "attention_blend": 0.15, "spatial_strength": 0.08},
    {"label": "alpha=0.05", "heat_alpha": 0.05, "attention_blend": 0.15, "spatial_strength": 0.08},
    {"label": "beta=0.05", "heat_alpha": 0.03, "attention_blend": 0.05, "spatial_strength": 0.08},
    {"label": "beta=0.15", "heat_alpha": 0.03, "attention_blend": 0.15, "spatial_strength": 0.08},
    {"label": "beta=0.30", "heat_alpha": 0.03, "attention_blend": 0.30, "spatial_strength": 0.08},
    {"label": "gamma=0.02", "heat_alpha": 0.03, "attention_blend": 0.15, "spatial_strength": 0.02},
    {"label": "gamma=0.08", "heat_alpha": 0.03, "attention_blend": 0.15, "spatial_strength": 0.08},
    {"label": "gamma=0.15", "heat_alpha": 0.03, "attention_blend": 0.15, "spatial_strength": 0.15},
]

sensitivity_rows = []
for idx, cfg in enumerate(sensitivity_settings):
    print(f"Testing sensitivity {cfg['label']}...")
    manager = make_manager(
        heat_alpha=cfg["heat_alpha"],
        attention_blend=cfg["attention_blend"],
        spatial_strength=cfg["spatial_strength"],
    )
    anchor_img, target_img, latency_s, _ = run_pair(
        cfg["label"],
        manager=manager,
        seed_base=2200 + idx * 10,
        boxes=region_boxes,
    )
    sensitivity_rows.append(metric_row(
        cfg["label"],
        target_img,
        anchor_img,
        latency_s,
        {
            "alpha": cfg["heat_alpha"],
            "beta": cfg["attention_blend"],
            "gamma": cfg["spatial_strength"],
        },
    ))

sensitivity_df = pd.DataFrame(sensitivity_rows)
display(Markdown("### MDCP Hyperparameter Sensitivity"))
display(sensitivity_df.sort_values(by="DINOv2", ascending=False))

paper_reference = pd.DataFrame([
    {"Config": "Baseline (No MDCP)", "Paper DINOv2": 0.582, "Paper LPIPS": 0.415},
    {"Config": "Core MDCP (L1+L2+L3)", "Paper DINOv2": 0.768, "Paper LPIPS": 0.252},
])
comparison_df = ablation_df.merge(paper_reference, on="Config", how="left")
comparison_df["DINOv2 Delta vs Paper"] = comparison_df["DINOv2"] - comparison_df["Paper DINOv2"]
comparison_df["LPIPS Delta vs Paper"] = comparison_df["LPIPS"] - comparison_df["Paper LPIPS"]
display(Markdown("### Comparison Against Paper Reference Values"))
display(comparison_df[["Config", "DINOv2", "Paper DINOv2", "DINOv2 Delta vs Paper", "LPIPS", "Paper LPIPS", "LPIPS Delta vs Paper"]])

NOTEBOOK_STATE["mdcp_ablation_results"] = ablation_df.to_dict(orient="records")
NOTEBOOK_STATE["mdcp_sensitivity_results"] = sensitivity_df.to_dict(orient="records")
NOTEBOOK_STATE["mdcp_examples"] = rendered_examples

backend.unload()
evaluator.free_memory()
print("MDCP benchmark complete.")


---

## 🚀 Phase 11: High-Density 50-Panel Single-Page Comic Generator (MDCP)

This section generates 50 distinct dramatic comic panel images using the core Multi-Level Diffusion Consistency Prior (MDCP) framework (`IntegratedComicPipeline`, `AdvancedAttentionManager`, `PanelEngine`, `MangaFlowLayoutEngine`, `ComicExporter`).

Designed for Kaggle GPU execution! When run on a Kaggle GPU (T4/P100), it automatically generates real SDXL panels for all 50 extreme action beats arranged on a single high-density comic page (2500x3750, 5x10 grid).

In [ ]:
# ============================================================
# 50-Panel Single-Page Comic Generation via MDCP Pipeline
# Optimized for Kaggle GPU & Local Execution
# ============================================================
import os, sys
import torch
from PIL import Image
from IPython.display import display

# Ensure paths are configured
sys.path.append(os.getcwd())
if os.path.exists(os.path.join(os.getcwd(), "indie_comic_pipeline")):
    sys.path.append(os.path.join(os.getcwd(), "indie_comic_pipeline"))

from run_50_panel_comic import run_50_panel_generation

# --- Interactive Custom Story Parameters ---
story_prompt = "Extreme Cyberpunk Action Saga" # @param {type:"string"}
character_name = "Wanderer" # @param {type:"string"}
story_world = "Earth & Orbit Battlefield" # @param {type:"string"}
force_dry_run = False # @param {type:"boolean"}

# Determine dry_run: if GPU is available on Kaggle, run real generation; otherwise fallback to dry-run
dry_run = force_dry_run or (not torch.cuda.is_available())

if torch.cuda.is_available() and not force_dry_run:
    print(f"⚡ GPU Detected: {torch.cuda.get_device_name(0)}. Running REAL SDXL + MDCP generation!")
else:
    print("💻 Running in DRY-RUN simulation mode.")

print(f"🚀 Executing MDCP 50-Panel Generation for '{story_prompt}'...")
results = run_50_panel_generation(
    prompt=story_prompt,
    character_name=character_name,
    story_world=story_world,
    dry_run=dry_run
)

print(f"\n🎉 50-Panel Single-Page Comic Generated Successfully!")
print(f" - High-Res Page (2500x3750): {results['page_path']}")
print(f" - Display Scale (1000x1500):  {results['display_path']}")
print(f" - CBZ Archive:               {results['cbz_path']}")
print(f" - PDF Document:              {results['pdf_path']}")
print(f" - Web Reader:                {results['html_path']}")

# Display assembled single-page 50-panel layout inline
display(results["page_image"])
